# Contract

A contract is the base concept from which you can calculate energy costs.
It consists of 4 components:
- a `provider` tariff
- a `distributor` tariff
- `fees` — which are essentially government tariffs — multiple fee tariffs can be provided as a list and will be merged automatically
- a `tax_rate` (a percentage applied to all tariff costs)

Given a contract and consumption data, you can calculate the cost of that consumption under the terms of the contract.

The output DataFrame uses a structured MultiIndex with fixed, translatable labels based on `TariffCategory` and `CostGroup` enums, making it predictable and easy to use for i18n.

In [1]:
from zoneinfo import ZoneInfo

import pandas as pd

from energy_cost import Contract, Meter, Tariff
from energy_cost.data.be import distributors, fees, tax_rate

contract = Contract(
    provider=Tariff.from_yaml("../examples/tariffs/fixed.yml"),
    distributor=distributors["fluvius_imewo"],
    fees=[fees["flanders_residential"], fees["be_residential"]],
    tax_rate=tax_rate,
    timezone=ZoneInfo("Europe/Brussels"),
)

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2024-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    )
)

contract.calculate_cost([consumption])

timestamp distributor                               \
                               capacity consumption            fixed   
                                  total       total data_manangement   
0 2024-01-01 00:00:00+01:00    8.209764   23.676520         1.209508   
1 2024-02-01 00:00:00+01:00    8.209764   22.149003         1.131475   
2 2024-03-01 00:00:00+01:00    8.209764    0.007956         1.207883   

                                      fees                                   \
                 total         consumption                            total   
      total      total energy_contribution     excise      total      total   
0  1.209508  33.095793            1.146415  28.260096  29.406511  29.406511   
1  1.131475  31.490243            1.072452  26.436864  27.509316  27.509316   
2  1.207883   9.425603            0.000385   0.009496   0.009881   0.009881   

     provider            taxes       total  
  consumption  total     total       total  
        total  total     total       total  
0       59.52  59.52  7.321338  129.343642  
1       55.68  55.68  6.880774  121.560333  
2        0.02   0.02  0.567329   10.022813

> Note: most tariffs are based on indexes, make sure to define all required indexes before calculating costs. See the [index documentation](./index.ipynb) for more details. If you are definingn your own tariffs, also have a look at the [tariff documentation](./tariff.ipynb) for details on how to implement tariffs.

### Resolution

By default the cost is calculated for each month, but you can specify a different resolution if you want to calculate costs for different time periods eg yearly.

In [2]:
import isodate

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    )
)

contract.calculate_cost([consumption], resolution=isodate.Duration(years=1))

timestamp distributor                                      \
                               capacity consumption            fixed          
                                  total       total data_manangement  total   
0 2024-01-01 00:00:00+01:00   98.517173  279.535692            14.28  14.28   
1 2025-01-01 00:00:00+01:00  133.105596  412.792925            17.51  17.51   
2 2026-01-01 00:00:00+01:00   33.875614   59.240491            17.85  17.85   

                             fees                                      \
        total         consumption                               total   
        total energy_contribution      excise       total       total   
0  392.332864           13.535090  333.651456  347.186546  347.186546   
1  563.408521           13.498109  332.739840  346.237949  346.237949   
2  110.966105            2.182271   53.794840   55.977111   55.977111   

     provider                               taxes        total  
  consumption     fixed          total      total        total  
        total fixed_fee  total   total      total        total  
0      702.72       0.0    0.0  702.72  86.534365  1528.773775  
1      630.72     120.0  120.0  750.72  99.621988  1759.988458  
2      135.96     180.0  180.0  315.96  28.974193   511.877409

### Time range

By default start and end are set to the first and last timestamp in the data, but you can specify a different start and end if you want to calculate costs for a different time period than the one covered by the data.

This is useful for the Flemish capacity tariff that uses a rolling average of the consumption of the last 12 months to determine the cost for the next month. In this case you would need to provide data for at least 12 months before the start of the period you want to calculate costs for.

In [3]:
import datetime as dt
from zoneinfo import ZoneInfo

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": [0.002] * 4 * 24 * 365 + [0.003] * 4 * 24 * 365 + [0.003] * 4 * 24 * 60 + [0.003],
        }
    )
)

contract.calculate_cost(
    [consumption],
    start=dt.datetime(2025, 1, 1, tzinfo=ZoneInfo("Europe/Brussels")),
    end=dt.datetime(2025, 12, 31, tzinfo=ZoneInfo("Europe/Brussels")),
)

timestamp distributor                               \
                                capacity consumption            fixed   
                                   total       total data_manangement   
0  2025-01-01 00:00:00+01:00   11.092133  525.886877         1.487151   
1  2025-02-01 00:00:00+01:00   11.092133  474.994598         1.343233   
2  2025-03-01 00:00:00+01:00   11.092133  525.180040         1.485152   
3  2025-04-01 00:00:00+02:00   11.092133  508.922784         1.439178   
4  2025-05-01 00:00:00+02:00   11.092133  525.886877         1.487151   
5  2025-06-01 00:00:00+02:00   11.104060  508.922784         1.439178   
6  2025-07-01 00:00:00+02:00   11.138350  525.886877         1.487151   
7  2025-08-01 00:00:00+02:00   11.192971  525.886877         1.487151   
8  2025-09-01 00:00:00+02:00   11.266127  508.922784         1.439178   
9  2025-10-01 00:00:00+02:00   11.356231  526.593714         1.489150   
10 2025-11-01 00:00:00+01:00   11.461871  508.922784         1.439178   
11 2025-12-01 00:00:00+01:00   11.461871  525.886877         1.487151   

                                        fees                          \
                   total         consumption                           
       total       total energy_contribution      excise       total   
0   1.487151  538.466160           17.196221  406.114744  423.310965   
1   1.343233  487.429964           15.532070  366.813317  382.345388   
2   1.485152  537.757324           17.173108  405.568891  422.741999   
3   1.439178  521.454095           16.641504  393.014268  409.655772   
4   1.487151  538.466160           17.196221  406.114744  423.310965   
5   1.439178  521.466022           16.641504  393.014268  409.655772   
6   1.487151  538.512378           17.196221  406.114744  423.310965   
7   1.487151  538.566998           17.196221  406.114744  423.310965   
8   1.439178  521.628089           16.641504  393.014268  409.655772   
9   1.489150  539.439095           17.219334  406.660597  423.879931   
10  1.439178  521.823833           16.641504  393.014268  409.655772   
11  1.487151  538.835898           17.196221  406.114744  423.310965   

                  provider                               taxes        total  
         total consumption     fixed         total       total        total  
         total       total fixed_fee total   total       total        total  
0   423.310965      803.52      10.0  10.0  813.52  106.517828  1881.814953  
1   382.345388      725.76      10.0  10.0  735.76   96.332121  1701.867473  
2   422.741999      802.44      10.0  10.0  812.44  106.376359  1879.315682  
3   409.655772      777.60      10.0  10.0  787.60  103.122592  1821.832460  
4   423.310965      803.52      10.0  10.0  813.52  106.517828  1881.814953  
5   409.655772      777.60      10.0  10.0  787.60  103.123308  1821.845102  
6   423.310965      803.52      10.0  10.0  813.52  106.520601  1881.863943  
7   423.310965      803.52      10.0  10.0  813.52  106.523878  1881.921841  
8   409.655772      777.60      10.0  10.0  787.60  103.133032  1822.016894  
9   423.879931      804.60      10.0  10.0  814.60  106.675142  1884.594168  
10  409.655772      777.60      10.0  10.0  787.60  103.144776  1822.224382  
11  423.310965      803.52      10.0  10.0  813.52  106.540012  1882.206875

As you can see in the above example, while the capacity was constant in 2025, the first months are still cheaper then the last, as the cost of the capacity tariff is based on the consumption of the previous 12 months, which includes the cheaper months of 2024.

### Injection

Aside from a consumption timeseries, you can also provide injection as a separate timeseries, as these are often measured independently and have different tariffs. If you provide injection data, the cost of the injection will be calculated separately and subtracted from the cost of the consumption, as injection is essentially negative consumption.

In [4]:
from energy_cost import PowerDirection

injection = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    ),
    direction=PowerDirection.INJECTION,
)

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    )
)

contract.calculate_cost([consumption, injection], resolution=isodate.Duration(years=1))

timestamp distributor                                      \
                               capacity consumption            fixed          
                                  total       total data_manangement  total   
0 2024-01-01 00:00:00+01:00   98.517173  279.535692            14.28  14.28   
1 2025-01-01 00:00:00+01:00  133.105596  412.792925            17.51  17.51   
2 2026-01-01 00:00:00+01:00   33.875614   59.240491            17.85  17.85   

                             fees                                      \
        total         consumption                               total   
        total energy_contribution      excise       total       total   
0  392.332864           13.535090  333.651456  347.186546  347.186546   
1  563.408521           13.498109  332.739840  346.237949  346.237949   
2  110.966105            2.182271   53.794840   55.977111   55.977111   

     provider                                          taxes        total  
  consumption     fixed        injection    total      total        total  
        total fixed_fee  total     total    total      total        total  
0      702.72       0.0    0.0  -140.544  562.176  78.101725  1379.797135  
1      630.72     120.0  120.0  -105.120  645.600  93.314788  1648.561258  
2      135.96     180.0  180.0   -28.325  287.635  27.274693   481.852909

### Expanding the MeterType level

By default the output collapses the `MeterType` level, summing across meter types (e.g. `single_rate`, `tou_peak`) for a cleaner 3-level output: `(TariffCategory, CostGroup, cost_type)`. If you need to distinguish between meter types, pass `expand_meter_types=True` to get a 4-level output: `(TariffCategory, CostGroup, MeterType, cost_type)`.

In [5]:
result = contract.calculate_cost(
    [consumption, injection], resolution=isodate.Duration(years=1), expand_meter_types=True
)
result

timestamp    provider                                        \
                            consumption   injection     fixed           total   
                            single_rate single_rate       all             all   
                                  total       total fixed_fee  total    total   
0 2024-01-01 00:00:00+01:00      702.72    -140.544       0.0    0.0  562.176   
1 2025-01-01 00:00:00+01:00      630.72    -105.120     120.0  120.0  645.600   
2 2026-01-01 00:00:00+01:00      135.96     -28.325     180.0  180.0  287.635   

  distributor                                                        fees  \
  consumption    capacity            fixed              total consumption   
  single_rate         all              all                all single_rate   
        total       total data_manangement  total       total      excise   
0  279.535692   98.517173            14.28  14.28  392.332864  333.651456   
1  412.792925  133.105596            17.51  17.51  563.408521  332.739840   
2   59.240491   33.875614            17.85  17.85  110.966105   53.794840   

                                                   taxes        total  
                                        total      total        total  
                                          all        all          all  
  energy_contribution       total       total      total        total  
0           13.535090  347.186546  347.186546  78.101725  1379.797135  
1           13.498109  346.237949  346.237949  93.314788  1648.561258  
2            2.182271   55.977111   55.977111  27.274693   481.852909